# Hybrid Detector Pipeline (Project Notebook)

This notebook uses existing detector outputs and the trained model to generate hybrid predictions.

Pipeline steps:
1. Load baseline, RAGAS, selfcheck, and similarity JSONL files.
2. Convert raw RAGAS/similarity scores using thresholds.
3. Build model-ready features in this exact order: baseline, ragas, selfcheck, similarity.
4. Run predictions using `hybrid_detector_model.pkl`.

In [ ]:
from pathlib import Path

from backend.config import (
    DEFAULT_BASELINE_FILE,
    DEFAULT_RAGAS_FILE,
    DEFAULT_SELFCHECK_FILE,
    DEFAULT_SIMILARITY_FILE,
    DEFAULT_HUMAN_FILE,
    MODEL_PATH,
    RAGAS_THRESHOLD,
    SIMILARITY_THRESHOLD,
    WORKSPACE_ROOT,
    PROJECT_ROOT,
    MODEL_FEATURES,
 )
from backend.detector_pipeline import build_records_from_paths
from backend.model_service import HybridModelService

print(f"Workspace root : {WORKSPACE_ROOT}")
print(f"Project root   : {PROJECT_ROOT}")
print(f"Model path     : {MODEL_PATH}")
print(f"Thresholds     : ragas<{RAGAS_THRESHOLD}, similarity<{SIMILARITY_THRESHOLD}")
print(f"Model features : {MODEL_FEATURES}")

In [ ]:
records = build_records_from_paths(
    baseline_path=DEFAULT_BASELINE_FILE,
    ragas_path=DEFAULT_RAGAS_FILE,
    selfcheck_path=DEFAULT_SELFCHECK_FILE,
    similarity_path=DEFAULT_SIMILARITY_FILE,
    human_path=DEFAULT_HUMAN_FILE,
)

print(f"Merged records: {len(records)}")
records[0] if records else {}

In [ ]:
import pandas as pd

service = HybridModelService()
predictions = service.predict_records(records)

human_by_id = {x["id"]: x["human"] for x in records if "human" in x}
for row in predictions:
    if row["id"] in human_by_id:
        row["human"] = human_by_id[row["id"]]

results_df = pd.DataFrame(predictions)
results_df.head(10)

In [ ]:
total = len(predictions)
hallucinated = sum(x["hybrid_prediction"] for x in predictions)
not_hallucinated = total - hallucinated

if total and all("human" in x for x in predictions):
    accuracy = sum(int(x["human"] == x["hybrid_prediction"]) for x in predictions) / total
else:
    accuracy = None

print(f"Total            : {total}")
print(f"Hallucinated     : {hallucinated}")
print(f"Not hallucinated : {not_hallucinated}")
print(f"Accuracy vs human: {accuracy if accuracy is not None else 'N/A'}")

output_path = PROJECT_ROOT / "hybrid_predictions_from_notebook.csv"
results_df.to_csv(output_path, index=False)
print(f"Saved predictions to: {output_path}")